In [20]:
# libraries calling 
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col

# create the session 
spark = SparkSession.builder \
    .appName("Spark_Data_Processing_and_Optimization") \
    .getOrCreate() 

In [2]:
spark

In [3]:
# the connection set up lets move further for proceding the task
# read the data from the source
df = spark.read.csv(
    "../Data/source.csv",
    header=True,
    inferSchema=True
)

In [4]:
# date source chcek 
df.show()

+----------+-----------------+-----------+-------+----------+-------+---------+------+------+--------+
|product_id|     product_name|   category|  price|base_price|user_id|   status|amount|region|priority|
+----------+-----------------+-----------+-------+----------+-------+---------+------+------+--------+
|       101|   Wireless Mouse|Electronics|  799.0|    677.12|   1001|Completed|  1598| North|    High|
|       102|Bluetooth Speaker|Electronics| 2499.0|    2117.8|   1002|  Pending|  2499| South|  Medium|
|       103|     Office Chair|  Furniture| 5999.0|    5083.9|   NULL|Completed|  5999| North|     Low|
|       104|    Notebook Pack| Stationery|  299.0|    253.39|   1004|Completed|  1196|  East|    High|
|       105|        USB Cable|Electronics|  199.0|    168.64|   1005|Cancelled|   398|  West|     Low|
|       106|      LED Monitor|Electronics|12999.0|   11016.1|   1006|Completed| 12999| North|    High|
|       107|     Water Bottle|       Home|  499.0|    422.88|   NULL|Comp

In [5]:
# lets check the schema of the data set
df.printSchema() # automatically Worked not even specifying =

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



## Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

### Answer

Apache Spark follows a master-worker architecture where different components work together to execute distributed data processing tasks.

### 1. Driver

The Driver is the main process of a Spark. It creates the Spark Session, converts the user program into execution tasks, schedules jobs, and collects the final results from the executors.

**Responsibilities:**
- Creates Spark Session
- Converts code into jobs and stages
- Schedules tasks
- Coordinates execution
- Collects results

---

### 2. Cluster Manager

The Cluster Manager is responsible for managing the computing resources available in the cluster. It allocates CPU cores and memory to Spark applications and launches executors on worker nodes.

**Responsibilities:**
- Manages cluster resources
- Allocates memory and CPU
- Starts executors
- Monitors resource usage

---

### 3. Executor

Executors are worker processes that run on worker nodes. They execute the tasks assigned by the Driver and return the results after processing the data.

**Responsibilities:**
- Execute tasks
- Process partitions of data
- Store intermediate results
- Return results to the Driver

---

## Q2. How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

### Answer

Lazy Evaluation is one of best techniques in Apache Spark. Instead of executing each transformation immediately, Spark records all transformations and waits until an action is called. During this time, Spark builds a Directed Acyclic Graph (DAG) of all operations.

When an action such as `show()`, `collect()`, or `count()` is executed, Spark analyzes the DAG and creates an optimized execution plan.

### Benefits of Lazy Evaluation

- Reduces unnecessary computations.
- Combines multiple transformations into a single optimized execution plan.
- Minimizes data movement across the cluster.
- Improves execution speed and resource utilization.

### Example

Suppose we write the following transformations:

```python
filtered_df = df.filter(col("category") == "Electronics")
selected_df = filtered_df.select("product_id", "price")
```

At this stage, Spark does **not** execute these operations.

Execution begins only when an action is performed:

```python
selected_df.show()
```

Spark then executes the entire workflow efficiently as a single optimized job instead of running each transformation separately.

## Q3. Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [6]:
# Read the CSV file

# as we need that 1 row must be treated as the header and inferSchema both as true as per the things mentioned the question

df = spark.read.csv(
    "../Data/source.csv",
    header=True,
    inferSchema=True
)

# Display the DataFrame
df.show(5)

+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+
|product_id|     product_name|   category| price|base_price|user_id|   status|amount|region|priority|
+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+
|       101|   Wireless Mouse|Electronics| 799.0|    677.12|   1001|Completed|  1598| North|    High|
|       102|Bluetooth Speaker|Electronics|2499.0|    2117.8|   1002|  Pending|  2499| South|  Medium|
|       103|     Office Chair|  Furniture|5999.0|    5083.9|   NULL|Completed|  5999| North|     Low|
|       104|    Notebook Pack| Stationery| 299.0|    253.39|   1004|Completed|  1196|  East|    High|
|       105|        USB Cable|Electronics| 199.0|    168.64|   1005|Cancelled|   398|  West|     Low|
+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+
only showing top 5 rows


## Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

### Answer

CSV and Parquet are two commonly used file formats in Apache Spark, but they differ in how data is stored and processed.

### CSV (Row-Based Storage)

CSV stores data row by row in plain text format. Each row contains all the values for a single record.

**Characteristics:**
- Row-based storage
- Human-readable text format
- Larger file size
- Slower read and write performance
- Does not store schema information
- Requires schema inference or manual schema definition

### Parquet (Columnar Storage)

Parquet stores data column by column in a binary format. Values from the same column are stored together, making it highly efficient for analytical workloads.

**Characteristics:**
- Columnar storage
- Compressed binary format
- Smaller file size
- Faster read performance
- Stores schema information
- Supports Predicate Pushdown and Column Pruning

---

### Conclusion

CSV is suitable for sharing and exchanging data because it is simple and readable. Parquet is the preferred format for Apache Spark applications because its columnar storage, compression, and built-in optimizations provide significantly better performance for large-scale data processing.

## Q5. Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the `category` is 'Electronics'.

In [7]:
# Select product_id and price for Electronics products

electronics_df = df.filter(col("category") == "Electronics").select("product_id", "price")

# Display the result
electronics_df.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|       101|  799.0|
|       102| 2499.0|
|       105|  199.0|
|       106|12999.0|
|       108| 1499.0|
|       110| 1099.0|
|       111| 3499.0|
|       114| 8999.0|
|       115| 1999.0|
+----------+-------+



## Q6. Write the code to revise a DataFrame by renaming the column `old_name` to `new_name` and casting the `price` column from a String to a Double.

In [8]:
# Rename the column 'product_name' to 'new_name'
# Cast the 'price' column from String to Double

revised_df = df.withColumnRenamed("product_name", "new_name") \
               .withColumn("price", col("price").cast("double"))

# Display the updated DataFrame
revised_df.show(5)

# Display the updated schema
revised_df.printSchema()

+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+
|product_id|         new_name|   category| price|base_price|user_id|   status|amount|region|priority|
+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+
|       101|   Wireless Mouse|Electronics| 799.0|    677.12|   1001|Completed|  1598| North|    High|
|       102|Bluetooth Speaker|Electronics|2499.0|    2117.8|   1002|  Pending|  2499| South|  Medium|
|       103|     Office Chair|  Furniture|5999.0|    5083.9|   NULL|Completed|  5999| North|     Low|
|       104|    Notebook Pack| Stationery| 299.0|    253.39|   1004|Completed|  1196|  East|    High|
|       105|        USB Cable|Electronics| 199.0|    168.64|   1005|Cancelled|   398|  West|     Low|
+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+
only showing top 5 rows
root
 |-- product_id: integer (nullable = true)
 |-- new_n

## Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

### Answer

Apache Spark provides fault tolerance through the **Lineage Graph**, also known as the **Directed Acyclic Graph (DAG)**. Instead of storing multiple copies of intermediate data, Spark records all the transformations applied to the data.

If a worker node or executor fails, Spark uses the Lineage Graph to identify the lost data partitions and recomputes only those partitions from the original data using the recorded transformations. This avoids reprocessing the entire dataset and improves the efficiency of recovery.

### Key Points

- Spark records every transformation in a Lineage Graph (DAG).
- The DAG maintains the sequence of operations performed on the data.
- If an executor fails, Spark identifies the lost partitions.
- Only the missing partitions are recomputed from the original data.
- This mechanism provides efficient fault tolerance without duplicating intermediate data.

---

## Q8. Write a query to filter a DataFrame `df_orders` for rows where the `status` is 'Completed' AND the `amount` is greater than 1000.

In [9]:
completed_orders= df.filter( (col("status") == "Completed") & (col ("amount") > 1000))
completed_orders.show()

+----------+--------------+-----------+-------+----------+-------+---------+------+------+--------+
|product_id|  product_name|   category|  price|base_price|user_id|   status|amount|region|priority|
+----------+--------------+-----------+-------+----------+-------+---------+------+------+--------+
|       101|Wireless Mouse|Electronics|  799.0|    677.12|   1001|Completed|  1598| North|    High|
|       103|  Office Chair|  Furniture| 5999.0|    5083.9|   NULL|Completed|  5999| North|     Low|
|       104| Notebook Pack| Stationery|  299.0|    253.39|   1004|Completed|  1196|  East|    High|
|       106|   LED Monitor|Electronics|12999.0|   11016.1|   1006|Completed| 12999| North|    High|
|       107|  Water Bottle|       Home|  499.0|    422.88|   NULL|Completed|  1497| South|  Medium|
|       108|  Laptop Stand|Electronics| 1499.0|   1270.34|   1008|Completed|  2998|  East|    High|
|       110|      Keyboard|Electronics| 1099.0|    931.36|   1010|Completed|  2198|  West|    High|


## Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

### Answer

Predicate Pushdown is a performance optimization technique used by Apache Spark when reading Parquet files. Instead of loading the entire dataset into memory, Spark pushes the filtering condition down to the Parquet storage layer.

As a result, only the rows that satisfy the specified condition are read from the disk, while the remaining data is skipped. This significantly reduces disk I/O, memory usage, and query execution time.

### Key Points

- Predicate Pushdown is supported by Parquet files.
- Spark applies filter conditions before loading the data into memory.
- Only the required rows are read from the storage.
- It reduces disk I/O and memory consumption.
- It improves the overall performance of Spark applications.

---

### Example

Suppose we want to retrieve only the completed orders:

```python
completed_orders = df.filter(col("status") == "Completed")
```

When the data is stored in **Parquet** format, Spark reads only the rows where the **status** is **Completed** instead of scanning the entire dataset.

---

### Benefits

- Faster query execution.
- Reduced memory usage.
- Less data transferred from disk.
- Better performance for large datasets.
- Efficient resource utilization.

### Conclusion

Predicate Pushdown is an important optimization feature of Parquet that allows Spark to read only the necessary data based on filter conditions. This minimizes the amount of data loaded into memory, resulting in faster and more efficient data processing.

## Q10. Write a code snippet to add a new column `final_price` which is the `base_price` multiplied by 1.18 (18% tax).

In [10]:
updated_df = df.withColumn("final Price", col("base_price") * 1.18)
updated_df.show(5)

+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+------------------+
|product_id|     product_name|   category| price|base_price|user_id|   status|amount|region|priority|       final Price|
+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+------------------+
|       101|   Wireless Mouse|Electronics| 799.0|    677.12|   1001|Completed|  1598| North|    High| 799.0015999999999|
|       102|Bluetooth Speaker|Electronics|2499.0|    2117.8|   1002|  Pending|  2499| South|  Medium|          2499.004|
|       103|     Office Chair|  Furniture|5999.0|    5083.9|   NULL|Completed|  5999| North|     Low|5999.0019999999995|
|       104|    Notebook Pack| Stationery| 299.0|    253.39|   1004|Completed|  1196|  East|    High|299.00019999999995|
|       105|        USB Cable|Electronics| 199.0|    168.64|   1005|Cancelled|   398|  West|     Low|198.99519999999998|
+----------+-----------------+--

## Q11. What is the difference between Transformations and Actions? Provide two examples of each.

### Answer

In Apache Spark, operations are divided into **Transformations** and **Actions**.

### Transformations

Transformations are operations that create a new DataFrame or RDD from an existing one. They are **lazy**, meaning Spark does not execute them immediately. Instead, Spark records these operations and executes them only when an action is called.

**Examples:**
- `filter()`
- `select()`

Example:

```python
filtered_df = df.filter(col("category") == "Electronics")
selected_df = filtered_df.select("product_id", "price")
```

The above code defines the transformations, but Spark does **not** execute them immediately.

---

### Actions

Actions are operations that trigger the execution of all pending transformations. They either return a result to the Driver or write data to storage.

**Examples:**
- `show()`
- `count()`

Example:

```python
filtered_df.show()
filtered_df.count()
```

When an action is executed, Spark processes all the previous transformations and returns the required result.

## Q12. Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where `user_id` is null, and save the result as a CSV at "path/to/output".

In [13]:
# Load the Parquet file
parquet_df = spark.read.parquet("../Data/parquet_output")

# Display the first 5 records
parquet_df.show(5)

# Filter rows where user_id is not null
filtered_df = parquet_df.filter(col("user_id").isNotNull())

# Display the filtered records
filtered_df.show()

# Save the filtered DataFrame as a CSV file
filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("../Data/csv_output")

print("Filtered data has been saved successfully as a CSV file.")

+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+
|product_id|     product_name|   category| price|base_price|user_id|   status|amount|region|priority|
+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+
|       101|   Wireless Mouse|Electronics| 799.0|    677.12| 1001.0|Completed|  1598| North|    High|
|       102|Bluetooth Speaker|Electronics|2499.0|    2117.8| 1002.0|  Pending|  2499| South|  Medium|
|       103|     Office Chair|  Furniture|5999.0|    5083.9|   NULL|Completed|  5999| North|     Low|
|       104|    Notebook Pack| Stationery| 299.0|    253.39| 1004.0|Completed|  1196|  East|    High|
|       105|        USB Cable|Electronics| 199.0|    168.64| 1005.0|Cancelled|   398|  West|     Low|
+----------+-----------------+-----------+------+----------+-------+---------+------+------+--------+
only showing top 5 rows
+----------+-----------------+-----------+-------+--------

## Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

### Answer

In Apache Spark, **Client Mode** and **Cluster Mode** are two different deployment modes that determine where the **Driver Program** runs.

### Client Mode

In **Client Mode**, the Driver Program runs on the local machine from which the Spark application is submitted, while the Executors run on the cluster. The client machine must remain connected throughout the execution of the application. This mode is mainly used for development, testing, and debugging because it allows easy monitoring of the application.

### Cluster Mode

In **Cluster Mode**, the Driver Program runs inside the cluster on one of the worker nodes, and the Executors also run on the cluster. Once the application is submitted, the client can disconnect because the cluster manages the entire execution. This mode is mainly used for production environments and large-scale data processing.

---

## Q14. Write a query to filter a dataset for rows where the `region` is **'North'** OR the `priority` is **'High'**.

In [19]:


# Filter rows where category is 'Electronics' OR status is 'Completed'

filtered_df = df.filter(
    (col("category") == "Electronics") |
    (col("status") == "Completed")
)

# Display the filtered records
filtered_df.show()

+----------+-----------------+-----------+-------+----------+-------+---------+------+------+--------+
|product_id|     product_name|   category|  price|base_price|user_id|   status|amount|region|priority|
+----------+-----------------+-----------+-------+----------+-------+---------+------+------+--------+
|       101|   Wireless Mouse|Electronics|  799.0|    677.12|   1001|Completed|  1598| North|    High|
|       102|Bluetooth Speaker|Electronics| 2499.0|    2117.8|   1002|  Pending|  2499| South|  Medium|
|       103|     Office Chair|  Furniture| 5999.0|    5083.9|   NULL|Completed|  5999| North|     Low|
|       104|    Notebook Pack| Stationery|  299.0|    253.39|   1004|Completed|  1196|  East|    High|
|       105|        USB Cable|Electronics|  199.0|    168.64|   1005|Cancelled|   398|  West|     Low|
|       106|      LED Monitor|Electronics|12999.0|   11016.1|   1006|Completed| 12999| North|    High|
|       107|     Water Bottle|       Home|  499.0|    422.88|   NULL|Comp

## Q15. When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

### Answer

When working with large datasets in Apache Spark, it is safer to use **`.show(5)`** instead of **`.collect()`** because `.show(5)` displays only the first five rows of the DataFrame, while `.collect()` retrieves the entire dataset from all worker nodes and sends it to the Driver Program.

For very large datasets, using `.collect()` can consume a large amount of memory on the Driver, which may lead to slow performance or an **Out of Memory (OOM)** error. In contrast, `.show(5)` retrieves only a small sample of the data, making it much faster and memory-efficient.

---

### Example

```python
# Display only the first 5 records
df.show(5)
```

```python
# Retrieve the entire dataset to the Driver
data = df.collect()
```

---